# Matched-null persistence barcodes: CEBRA versus PCA

This notebook rebuilds the comparison from cleaned neural recordings so that the real and null point clouds are directly matched.

For each region, stimulus, recording, and requested embedding dimension it:

1. standardizes neurons using statistics from the real recording;
2. fits CEBRA and PCA once to the real recording;
3. creates 100 null realizations by independently circularly shifting each neuron within each trial;
4. transforms each null through the **same fitted representation** as the real data;
5. uses the same sampled time indices for real/null and CEBRA/PCA point clouds;
6. centers each point cloud and divides it by its RMS radius (unit-scale normalization);
7. computes sparse Vietoris-Rips persistence in H0, H1, and H2;
8. compares each real barcode with a representative null barcode and with the full 100-shuffle null distribution of maximum persistence;
9. reports the number of real features exceeding the recording-specific null threshold.

The output includes one six-panel barcode figure per recording/stimulus (CEBRA and PCA by H0-H2), plus regional summary figures for each homology group. Real bars exceeding the recording-specific 95th percentile of the null maximum bar lifetime are highlighted.

> **Runtime:** the full analysis is large. Fitted reducers, matched sample indices, every PH computation, metric tables, and completion metadata are checkpointed so interrupted runs can be resumed. Start with `SMOKE_TEST = True`, then set it to `False` for the complete analysis.

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import pickle
import random
import re
import sys
import warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from ripser import ripser
from scipy.sparse import csr_matrix
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

try:
    from cebra import CEBRA
except ImportError as exc:
    raise ImportError(
        "CEBRA is required. Run this notebook in the repository's topology environment."
    ) from exc

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tda_utils import TDADataManager

sns.set_theme(style="ticks", context="notebook")
plt.rcParams["figure.dpi"] = 140

## Configuration

`N_POINTS` is deliberately global: every point cloud uses exactly this many points. The same time indices are reused across methods and all null realizations for a recording. `K_NEIGHBORS` is smaller than the legacy notebook's 300-neighbor graph to keep 100 H2 null computations per recording tractable; increase it only after a sensitivity check.

Each barcode panel displays the `BARCODE_TOP_N` longest finite intervals. The representative null is the shuffle whose maximum persistence is closest to that recording's median null maximum. A real bar is highlighted only when its lifetime exceeds the `NULL_QUANTILE` quantile of the maximum lifetime across all null diagrams for that recording, method, and homology group.

In [ ]:
SEED = 42
REGIONS = ("LGN", "V1")
STIMULI = (
    "color_exchange",
    "luminance_flash",
    "drifting_gratings",
    "chromatic_gratings",
)
EMBED_DIMS = (3,)
METHODS = ("CEBRA", "PCA")

NULL_TYPE = "within_trial_circular_shift"
N_SHUFFLES = 100
N_POINTS = 1_000
TRIAL_LENGTH = 1_000  # samples; validated against every recording below
K_NEIGHBORS = 50
MAX_HOMOLOGY = 2
BARCODE_TOP_N = 50
NULL_QUANTILE = 0.95
SAVE_RECORDING_PDF = False

CEBRA_MAX_ITERATIONS = 30_000
CEBRA_BATCH_SIZE = 512
CEBRA_TIME_OFFSETS = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Full run by default. Set True to validate the workflow on one condition,
# one dimension, and two null draws before committing substantial compute.
SMOKE_TEST = False
SMOKE_REGION = "LGN"
SMOKE_STIMULUS = "color_exchange"
SMOKE_RECORDING_LIMIT = 1

CACHE_TAG = (
    f"within_trial_cshift_v2_n{N_POINTS}_k{K_NEIGHBORS}_"
    f"s{N_SHUFFLES}_cebra{CEBRA_MAX_ITERATIONS}"
)
CACHE_ROOT = REPO_ROOT / "data" / "analysis_results" / "matched_null_cache" / CACHE_TAG
FIGURE_ROOT = REPO_ROOT / "data" / "analysis_results" / "matched_null_barcodes"
MODEL_ROOT = CACHE_ROOT / "fitted_reducers"
INDEX_ROOT = CACHE_ROOT / "matched_indices"
COMPLETION_ROOT = CACHE_ROOT / "completed_conditions"
INTERMEDIATE_ROOT = FIGURE_ROOT / "intermediate"
for directory in (CACHE_ROOT, FIGURE_ROOT, MODEL_ROOT, INDEX_ROOT, COMPLETION_ROOT, INTERMEDIATE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

run_config = {
    "seed": SEED,
    "regions": REGIONS,
    "stimuli": STIMULI,
    "embedding_dimensions": EMBED_DIMS,
    "methods": METHODS,
    "null_type": NULL_TYPE,
    "n_shuffles": N_SHUFFLES,
    "n_points": N_POINTS,
    "trial_length": TRIAL_LENGTH,
    "k_neighbors": K_NEIGHBORS,
    "max_homology": MAX_HOMOLOGY,
    "cebra_max_iterations": CEBRA_MAX_ITERATIONS,
    "device": DEVICE,
    "cache_tag": CACHE_TAG,
}
config_path = INTERMEDIATE_ROOT / "analysis_config.json"
config_temporary = config_path.with_suffix(".tmp")
config_temporary.write_text(json.dumps(run_config, indent=2), encoding="utf-8")
config_temporary.replace(config_path)

print(f"Device: {DEVICE}")
print(f"Cache: {CACHE_ROOT}")

## Discover real recordings

Only the unshuffled cleaned files are loaded. Nulls are regenerated here by circularly shifting each neuron independently within each inferred 1,000-sample trial. Trial divisibility is checked before analysis. A length mismatch in a source file is reported and the same `zip(recordings, datas)` behavior used by the original notebooks is retained.

In [ ]:
data_manager = TDADataManager()
clean_dir = data_manager.data_paths["clean_spike_data_dir"]


def parse_dataset_name(path: Path) -> tuple[str, str]:
    stem = path.stem
    region, stimulus = stem.split("_", 1)
    return region, stimulus


dataset_paths = {}
for path in sorted(clean_dir.glob("*.pkl")):
    if "shuffled" in path.stem:
        continue
    region, stimulus = parse_dataset_name(path)
    if region in REGIONS and stimulus in STIMULI:
        dataset_paths[(region, stimulus)] = path

manifest_rows = []
for (region, stimulus), path in sorted(dataset_paths.items()):
    with path.open("rb") as handle:
        payload = pickle.load(handle)
    recordings = list(payload["recordings"])
    datas = list(payload["datas"])
    if len(recordings) != len(datas):
        warnings.warn(
            f"{path.name}: {len(recordings)} labels but {len(datas)} matrices; "
            "only paired entries will be analyzed."
        )
    for recording, matrix in zip(recordings, datas):
        manifest_rows.append(
            {
                "region": region,
                "stimulus": stimulus,
                "recording": str(recording),
                "time_points": int(matrix.shape[0]),
                "neurons": int(matrix.shape[1]),
                "path": path,
            }
        )
    del payload, datas
    gc.collect()

manifest = pd.DataFrame(manifest_rows)
invalid_trial_lengths = manifest[manifest.time_points % TRIAL_LENGTH != 0]
if not invalid_trial_lengths.empty:
    raise ValueError(
        "TRIAL_LENGTH does not evenly divide these recordings:\n"
        + invalid_trial_lengths[["region", "stimulus", "recording", "time_points"]].to_string(index=False)
    )
manifest["n_trials"] = manifest.time_points // TRIAL_LENGTH
if SMOKE_TEST:
    manifest = manifest[
        (manifest.region == SMOKE_REGION)
        & (manifest.stimulus == SMOKE_STIMULUS)
    ].groupby(["region", "stimulus"], as_index=False).head(SMOKE_RECORDING_LIMIT)

manifest_for_export = manifest.copy()
manifest_for_export["path"] = manifest_for_export["path"].astype(str)
manifest_for_export.to_csv(INTERMEDIATE_ROOT / "recording_manifest.csv", index=False)
display(manifest.drop(columns="path"))

## Embedding and null helpers

Each 1,000-sample trial is shifted separately, and every neuron receives an independently sampled nonzero offset within that trial. Reducers are fitted only on the real standardized data; null data are transformed through those fixed maps, avoiding 100 refits and keeping the null focused on coordinated within-trial timing.

In [ ]:
def stable_seed(*parts) -> int:
    token = "|".join(map(str, parts)).encode("utf-8")
    return int.from_bytes(hashlib.sha256(token).digest()[:4], "little")


def as_numpy(matrix) -> np.ndarray:
    if torch.is_tensor(matrix):
        matrix = matrix.detach().cpu().numpy()
    return np.asarray(matrix, dtype=np.float32)


def within_trial_circular_shift(
    X: np.ndarray, rng: np.random.Generator, trial_length: int = TRIAL_LENGTH
) -> np.ndarray:
    '''Independently circularly shift every neuron inside every trial.'''
    n_time, n_neurons = X.shape
    if n_time % trial_length != 0:
        raise ValueError(
            f"{n_time} samples cannot be divided into {trial_length}-sample trials."
        )
    n_trials = n_time // trial_length
    trials = X.reshape(n_trials, trial_length, n_neurons)
    shifted = np.empty_like(trials)
    offsets = rng.integers(1, trial_length, size=(n_trials, n_neurons))
    time = np.arange(trial_length)[:, None]
    neurons = np.arange(n_neurons)[None, :]
    for trial in range(n_trials):
        source_time = (time - offsets[trial][None, :]) % trial_length
        shifted[trial] = trials[trial][source_time, neurons]
    return shifted.reshape(X.shape)


def matched_indices(n_time: int, key: tuple) -> np.ndarray:
    if n_time < N_POINTS:
        raise ValueError(f"Recording has {n_time} points, fewer than N_POINTS={N_POINTS}.")
    rng = np.random.default_rng(stable_seed(SEED, "indices", *key))
    return np.sort(rng.choice(n_time, size=N_POINTS, replace=False))


def unit_rms_point_cloud(Y: np.ndarray) -> np.ndarray:
    '''Center a point cloud and divide by its root-mean-square radius.'''
    Y = np.asarray(Y, dtype=np.float32)
    Y = Y - Y.mean(axis=0, keepdims=True)
    radius = float(np.sqrt(np.mean(np.sum(Y * Y, axis=1))))
    if not np.isfinite(radius) or radius <= np.finfo(np.float32).eps:
        raise ValueError("Cannot scale-normalize a degenerate point cloud.")
    return Y / radius


def make_cebra(dim: int, seed: int):
    torch.manual_seed(seed)
    return CEBRA(
        model_architecture="offset10-model",
        batch_size=CEBRA_BATCH_SIZE,
        learning_rate=3e-4,
        temperature=1,
        output_dimension=dim,
        max_iterations=CEBRA_MAX_ITERATIONS,
        num_hidden_units=128,
        distance="cosine",
        conditional="time",
        device=DEVICE,
        verbose=False,
        time_offsets=CEBRA_TIME_OFFSETS,
        optimizer="adam",
    )


def fit_reducers(X_real: np.ndarray, dim: int, seed: int):
    scaler = StandardScaler().fit(X_real)
    X_scaled = scaler.transform(X_real).astype(np.float32, copy=False)

    pca = PCA(n_components=dim, svd_solver="randomized", random_state=seed)
    pca.fit(X_scaled)

    cebra_model = make_cebra(dim, seed)
    cebra_model.fit(X_scaled)
    return scaler, {"PCA": pca, "CEBRA": cebra_model}


def transform_cloud(
    X: np.ndarray,
    scaler: StandardScaler,
    reducer,
    sample_idx: np.ndarray,
) -> np.ndarray:
    X_scaled = scaler.transform(X).astype(np.float32, copy=False)
    embedding = reducer.transform(X_scaled)
    return unit_rms_point_cloud(np.asarray(embedding)[sample_idx])

## Persistent homology and checkpointing

PH uses an undirected union-symmetrized Euclidean kNN graph. The filtration threshold is unrestricted; all finite sparse-graph edges can enter. Every PH cache record contains all three diagrams and complete provenance.

Matched sample indices, the fitted scaler/PCA model, and the fitted CEBRA model are also saved per recording and dimension. If a run stops partway through a recording, rerunning this notebook loads those fitted objects and computes only missing null diagrams.

In [ ]:
def sparse_knn_distance(point_cloud: np.ndarray) -> csr_matrix:
    n = point_cloud.shape[0]
    k = min(K_NEIGHBORS, n - 1)
    nn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean", n_jobs=-1)
    distances, neighbors = nn.fit(point_cloud).kneighbors(point_cloud)

    rows = np.repeat(np.arange(n), k)
    cols = neighbors[:, 1 : k + 1].reshape(-1)
    vals = distances[:, 1 : k + 1].astype(np.float32).reshape(-1)
    graph = csr_matrix((vals, (rows, cols)), shape=(n, n))
    graph = graph.maximum(graph.T)
    graph.setdiag(0)
    graph.eliminate_zeros()
    return graph


def persistence_diagrams(point_cloud: np.ndarray) -> list[np.ndarray]:
    graph = sparse_knn_distance(point_cloud)
    return ripser(
        graph,
        distance_matrix=True,
        maxdim=MAX_HOMOLOGY,
        thresh=np.inf,
        coeff=2,
    )["dgms"]


def result_path(meta: dict) -> Path:
    state = "real" if meta["null_index"] is None else f"null_{meta['null_index']:03d}"
    name = "__".join(
        [
            meta["region"],
            meta["stimulus"],
            meta["recording"],
            f"d{meta['dimension']}",
            meta["method"].lower(),
            state,
        ]
    )
    return CACHE_ROOT / f"{name}.pkl"


def save_result(path: Path, payload: dict) -> None:
    temporary = path.with_suffix(".tmp")
    with temporary.open("wb") as handle:
        pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
    temporary.replace(path)



def condition_stem(meta: dict, include_dimension: bool = True) -> str:
    parts = [meta["region"], meta["stimulus"], str(meta["recording"])]
    if include_dimension:
        parts.append(f"d{meta['dimension']}")
    return "__".join(re.sub(r"[^A-Za-z0-9_.-]+", "_", part) for part in parts)


def reducer_paths(base_meta: dict) -> tuple[Path, Path]:
    stem = condition_stem(base_meta)
    return MODEL_ROOT / f"{stem}__preprocess.pkl", MODEL_ROOT / f"{stem}__cebra.pt"


def save_fitted_reducers(base_meta: dict, scaler, reducers: dict) -> None:
    preprocess_path, cebra_path = reducer_paths(base_meta)
    preprocess_temporary = preprocess_path.with_suffix(".tmp")
    with preprocess_temporary.open("wb") as handle:
        pickle.dump(
            {"scaler": scaler, "pca": reducers["PCA"]},
            handle,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
    preprocess_temporary.replace(preprocess_path)

    cebra_temporary = cebra_path.with_suffix(".tmp")
    reducers["CEBRA"].save(str(cebra_temporary))
    cebra_temporary.replace(cebra_path)


def fit_or_load_reducers(X_real: np.ndarray, dim: int, seed: int, base_meta: dict):
    preprocess_path, cebra_path = reducer_paths(base_meta)
    if preprocess_path.exists() and cebra_path.exists():
        with preprocess_path.open("rb") as handle:
            preprocess = pickle.load(handle)
        cebra_model = CEBRA.load(str(cebra_path), map_location=DEVICE).to(DEVICE)
        print(f"Loaded fitted reducers: {condition_stem(base_meta)}")
        return preprocess["scaler"], {"PCA": preprocess["pca"], "CEBRA": cebra_model}

    scaler, reducers = fit_reducers(X_real, dim, seed)
    save_fitted_reducers(base_meta, scaler, reducers)
    return scaler, reducers


def load_or_create_matched_indices(n_time: int, key: tuple) -> np.ndarray:
    meta = {"region": key[0], "stimulus": key[1], "recording": key[2]}
    path = INDEX_ROOT / f"{condition_stem(meta, include_dimension=False)}.npy"
    if path.exists():
        indices = np.load(path)
        if len(indices) != N_POINTS or np.any(indices >= n_time):
            raise ValueError(f"Cached matched indices are incompatible with {path}.")
        return indices

    indices = matched_indices(n_time, key)
    temporary = path.with_suffix(".tmp")
    with temporary.open("wb") as handle:
        np.save(handle, indices)
    temporary.replace(path)
    return indices


def mark_condition_complete(base_meta: dict, paths: list[Path]) -> None:
    if not all(path.exists() for path in paths):
        return
    marker = COMPLETION_ROOT / f"{condition_stem(base_meta)}.json"
    temporary = marker.with_suffix(".tmp")
    payload = {
        **base_meta,
        "n_result_files": len(paths),
        "result_files": [path.name for path in paths],
    }
    temporary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    temporary.replace(marker)


def write_cache_inventory() -> None:
    rows = [
        {
            "file": path.name,
            "size_bytes": path.stat().st_size,
            "modified_ns": path.stat().st_mtime_ns,
        }
        for path in sorted(CACHE_ROOT.glob("*.pkl"))
    ]
    pd.DataFrame(rows).to_csv(INTERMEDIATE_ROOT / "ph_cache_inventory.csv", index=False)

def expected_paths(base_meta: dict, n_shuffles: int) -> list[Path]:
    paths = []
    for method in METHODS:
        meta = {**base_meta, "method": method, "null_index": None}
        paths.append(result_path(meta))
        for null_index in range(n_shuffles):
            meta = {**base_meta, "method": method, "null_index": null_index}
            paths.append(result_path(meta))
    return paths

## Run the matched analysis

This cell is resumable at three levels: matched sample indices, fitted reducers, and individual PH diagrams. A recording/dimension is skipped when every expected real and null result is present; a partially completed condition loads its fitted models and computes only missing diagrams. PCA dimensions larger than the number of recorded neurons are skipped for both methods so that PCA and CEBRA remain paired.

In [ ]:
active_dims = (3,) if SMOKE_TEST else EMBED_DIMS
active_shuffles = 2 if SMOKE_TEST else N_SHUFFLES

for dataset_key, dataset_manifest in manifest.groupby(["region", "stimulus"], sort=True):
    region, stimulus = dataset_key
    path = dataset_manifest.iloc[0]["path"]
    with Path(path).open("rb") as handle:
        payload = pickle.load(handle)
    matrix_by_recording = {
        str(recording): as_numpy(matrix)
        for recording, matrix in zip(payload["recordings"], payload["datas"])
    }
    del payload

    for row in dataset_manifest.itertuples(index=False):
        recording = row.recording
        X_real = matrix_by_recording[recording]
        key = (region, stimulus, recording)
        sample_idx = load_or_create_matched_indices(X_real.shape[0], key)

        for dim in active_dims:
            if dim > X_real.shape[1]:
                print(
                    f"Skip {region}/{stimulus}/{recording}/{dim}d: "
                    f"PCA has only {X_real.shape[1]} neural features."
                )
                continue

            base_meta = {
                "region": region,
                "stimulus": stimulus,
                "recording": recording,
                "dimension": dim,
                "n_points": N_POINTS,
                "k_neighbors": K_NEIGHBORS,
                "null_type": NULL_TYPE,
            }
            condition_paths = expected_paths(base_meta, active_shuffles)
            if all(path.exists() for path in condition_paths):
                mark_condition_complete(base_meta, condition_paths)
                print(f"Cached: {region}/{stimulus}/{recording}/{dim}d")
                continue

            fit_seed = stable_seed(SEED, "fit", region, stimulus, recording, dim)
            scaler, reducers = fit_or_load_reducers(X_real, dim, fit_seed, base_meta)

            for method, reducer in reducers.items():
                meta = {**base_meta, "method": method, "null_index": None}
                path_out = result_path(meta)
                if not path_out.exists():
                    cloud = transform_cloud(X_real, scaler, reducer, sample_idx)
                    save_result(path_out, {"meta": meta, "dgms": persistence_diagrams(cloud)})

            for null_index in range(active_shuffles):
                null_seed = stable_seed(
                    SEED, NULL_TYPE, region, stimulus, recording, null_index
                )
                rng = np.random.default_rng(null_seed)
                X_null = within_trial_circular_shift(X_real, rng)
                for method, reducer in reducers.items():
                    meta = {**base_meta, "method": method, "null_index": null_index}
                    path_out = result_path(meta)
                    if path_out.exists():
                        continue
                    cloud = transform_cloud(X_null, scaler, reducer, sample_idx)
                    save_result(path_out, {"meta": meta, "dgms": persistence_diagrams(cloud)})
                del X_null
                gc.collect()

            mark_condition_complete(base_meta, condition_paths)
            del scaler, reducers
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

    del matrix_by_recording
    gc.collect()

write_cache_inventory()
print("PH cache inventory updated.")

## Extract barcode statistics

Infinite H0 classes and zero-length intervals are removed. For every real and shuffled diagram, the table records feature count, total persistence, mean persistence, and maximum persistence. The inferential comparison uses maximum persistence because it provides a barcode-native, familywise threshold for deciding whether any real interval is longer than expected under the null.

In [ ]:
def finite_diagrams(dgms) -> list[np.ndarray]:
    cleaned = []
    for homology in range(MAX_HOMOLOGY + 1):
        dgm = np.asarray(dgms[homology], dtype=float)
        if dgm.size == 0:
            cleaned.append(np.empty((0, 2), dtype=float))
            continue
        keep = np.isfinite(dgm).all(axis=1) & (dgm[:, 1] > dgm[:, 0])
        cleaned.append(dgm[keep])
    return cleaned


def diagram_metrics(diagram: np.ndarray) -> dict:
    lifetimes = diagram[:, 1] - diagram[:, 0] if len(diagram) else np.array([])
    return {
        "feature_count": int(len(lifetimes)),
        "total_persistence": float(lifetimes.sum()) if len(lifetimes) else 0.0,
        "mean_persistence": float(lifetimes.mean()) if len(lifetimes) else 0.0,
        "max_persistence": float(lifetimes.max()) if len(lifetimes) else 0.0,
    }


def current_result_files(dim: int) -> list[Path]:
    return sorted(CACHE_ROOT.glob(f"*__d{dim}__*.pkl"))


def load_barcode_metrics(dim: int) -> pd.DataFrame:
    files = current_result_files(dim)
    if not files:
        raise FileNotFoundError(f"No cached results found for {dim}d.")

    table_path = INTERMEDIATE_ROOT / f"barcode_metrics_{dim}d.csv"
    expected_paths = {str(path) for path in files}
    if table_path.exists():
        cached = pd.read_csv(table_path)
        cached_paths = set(cached.get("result_path", pd.Series(dtype=str)).astype(str))
        if len(cached) == len(files) * (MAX_HOMOLOGY + 1) and cached_paths == expected_paths:
            return cached

    rows = []
    for path in files:
        with path.open("rb") as handle:
            result = pickle.load(handle)
        meta = result["meta"]
        state = "real" if meta["null_index"] is None else "within-trial shift"
        for homology, diagram in enumerate(finite_diagrams(result["dgms"])):
            rows.append(
                {
                    **meta,
                    "state": state,
                    "homology": homology,
                    "result_path": str(path),
                    **diagram_metrics(diagram),
                }
            )
    metrics = pd.DataFrame(rows)
    temporary = table_path.with_suffix(".tmp")
    metrics.to_csv(temporary, index=False)
    temporary.replace(table_path)
    return metrics


def recording_null_summary(metrics: pd.DataFrame) -> pd.DataFrame:
    group_cols = [
        "dimension",
        "method",
        "region",
        "stimulus",
        "recording",
        "homology",
    ]
    rows = []
    for key, group in metrics.groupby(group_cols, sort=True):
        real_rows = group.loc[group.state == "real"]
        null = group.loc[group.state == "within-trial shift", "max_persistence"]
        if len(real_rows) != 1 or null.empty:
            continue
        real_row = real_rows.iloc[0]
        real_value = float(real_row.max_persistence)
        null_values = null.to_numpy(float)
        null_sd = float(np.std(null_values, ddof=1)) if len(null_values) > 1 else np.nan
        null_threshold = float(np.quantile(null_values, NULL_QUANTILE))
        with Path(real_row.result_path).open("rb") as handle:
            real_diagram = finite_diagrams(pickle.load(handle)["dgms"])[int(key[-1])]
        real_lifetimes = real_diagram[:, 1] - real_diagram[:, 0]
        n_features_above_null = int(np.sum(real_lifetimes > null_threshold))
        rows.append(
            {
                **dict(zip(group_cols, key)),
                "real_max_persistence": real_value,
                "null_mean": float(np.mean(null_values)),
                "null_low": float(np.quantile(null_values, 0.025)),
                "null_high": float(np.quantile(null_values, 0.975)),
                "null_q95_max": null_threshold,
                "n_features_above_null": n_features_above_null,
                "null_sd": null_sd,
                "z_score": (real_value - float(np.mean(null_values))) / null_sd
                if np.isfinite(null_sd) and null_sd > 0
                else np.nan,
                "p_upper": float(
                    (1 + np.sum(null_values >= real_value)) / (len(null_values) + 1)
                ),
                "n_null": len(null_values),
            }
        )
    return pd.DataFrame(rows)

## Barcode figures and regional null summaries

Each recording-level panel shows the longest finite intervals from the real diagram above zero and a representative null diagram below zero. The representative is the shuffle nearest the median null maximum persistence; it is for visualization only. Dark real bars pass the recording-specific 95th percentile of the **maximum** bar lifetime across all null shuffles.

The regional summary retains every recording as the observational unit. Filled markers are real maximum persistence, open markers are null means, and vertical lines are the central 95% of the 100-shuffle null distribution.

In [ ]:
REGION_COLORS = {"LGN": "#0072B2", "V1": "#D55E00"}
STIMULUS_LABELS = {
    "color_exchange": "Color\nexchange",
    "luminance_flash": "Luminance\nflash",
    "drifting_gratings": "Drifting\ngratings",
    "chromatic_gratings": "Chromatic\ngratings",
}


def load_diagrams(path: str) -> list[np.ndarray]:
    with Path(path).open("rb") as handle:
        return finite_diagrams(pickle.load(handle)["dgms"])


def longest_intervals(diagram: np.ndarray, n: int = BARCODE_TOP_N) -> np.ndarray:
    if len(diagram) == 0:
        return diagram
    lifetimes = diagram[:, 1] - diagram[:, 0]
    return diagram[np.argsort(lifetimes)[::-1][:n]]


def draw_barcode_pair(
    ax,
    real_diagram: np.ndarray,
    null_diagram: np.ndarray,
    threshold: float,
    color: str,
):
    all_real_lifetimes = (
        real_diagram[:, 1] - real_diagram[:, 0] if len(real_diagram) else np.array([])
    )
    n_features_above_null = int(np.sum(all_real_lifetimes > threshold))
    real = longest_intervals(real_diagram)
    null = longest_intervals(null_diagram)
    shown_lifetimes = real[:, 1] - real[:, 0] if len(real) else np.array([])
    significant = shown_lifetimes > threshold

    if len(real):
        real_y = np.arange(1, len(real) + 1)
        real_colors = np.where(significant, color, "#9AA0A6")
        real_widths = np.where(significant, 2.4, 1.15)
        ax.hlines(real_y, real[:, 0], real[:, 1], colors=real_colors, linewidths=real_widths)
    if len(null):
        null_y = -np.arange(1, len(null) + 1)
        ax.hlines(null_y, null[:, 0], null[:, 1], color="#666666", linewidth=1.05)

    all_intervals = [part for part in (real, null) if len(part)]
    max_death = max((float(part[:, 1].max()) for part in all_intervals), default=1.0)
    ax.axhline(0, color="0.82", linewidth=0.8)
    ax.set_xlim(0, max_death * 1.04)
    ax.set_ylim(-(max(len(null), 1) + 2), max(len(real), 1) + 2)
    ax.set_yticks([])
    ax.text(0.01, 0.96, "Real", transform=ax.transAxes, va="top", fontsize=8)
    ax.text(0.01, 0.04, "Representative null", transform=ax.transAxes, va="bottom", fontsize=8)
    ax.text(
        0.99,
        0.96,
        f"Features above null: {n_features_above_null}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=9,
        fontweight="bold",
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.8, "pad": 1.5},
    )


def representative_null(group: pd.DataFrame) -> pd.Series:
    target = float(group["max_persistence"].median())
    index = (group["max_persistence"] - target).abs().idxmin()
    return group.loc[index]


def safe_filename(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def plot_recording_barcodes(condition: pd.DataFrame, dim: int):
    identity = condition.iloc[0]
    fig, axes = plt.subplots(
        len(METHODS),
        MAX_HOMOLOGY + 1,
        figsize=(15, 8),
        constrained_layout=True,
    )
    diagram_cache = {}

    for row, method in enumerate(METHODS):
        for homology in range(MAX_HOMOLOGY + 1):
            ax = axes[row, homology]
            panel = condition[
                (condition.method == method) & (condition.homology == homology)
            ]
            real_rows = panel[panel.state == "real"]
            null_rows = panel[panel.state == "within-trial shift"]
            if len(real_rows) != 1 or null_rows.empty:
                ax.text(0.5, 0.5, "Incomplete cache", ha="center", va="center")
                ax.set_axis_off()
                continue

            real_row = real_rows.iloc[0]
            null_row = representative_null(null_rows)
            for result_path in (real_row.result_path, null_row.result_path):
                if result_path not in diagram_cache:
                    diagram_cache[result_path] = load_diagrams(result_path)
            threshold = float(np.quantile(null_rows.max_persistence, NULL_QUANTILE))
            draw_barcode_pair(
                ax,
                diagram_cache[real_row.result_path][homology],
                diagram_cache[null_row.result_path][homology],
                threshold,
                REGION_COLORS[identity.region],
            )
            ax.set_title(
                f"H{homology}  |  null #{int(null_row.null_index):02d}", fontsize=10
            )
            if homology == 0:
                ax.set_ylabel(method)
            if row == len(METHODS) - 1:
                ax.set_xlabel("Filtration value")

    fig.suptitle(
        f"{identity.region} | {identity.stimulus} | {identity.recording} | {dim}D",
        fontsize=15,
        fontweight="bold",
    )
    return fig


def plot_regional_summary(summary: pd.DataFrame, dim: int, homology: int):
    fig, axes = plt.subplots(
        len(METHODS),
        len(REGIONS),
        figsize=(13, 8),
        constrained_layout=True,
        squeeze=False,
    )
    subset_h = summary[summary.homology == homology]
    for row, method in enumerate(METHODS):
        for col, region in enumerate(REGIONS):
            ax = axes[row, col]
            subset = subset_h[(subset_h.method == method) & (subset_h.region == region)]
            for stim_index, stimulus in enumerate(STIMULI):
                stim_data = subset[subset.stimulus == stimulus].sort_values("recording")
                if stim_data.empty:
                    continue
                offsets = (
                    np.linspace(-0.22, 0.22, len(stim_data))
                    if len(stim_data) > 1
                    else [0.0]
                )
                for offset, record in zip(offsets, stim_data.itertuples(index=False)):
                    x = stim_index + offset
                    ax.vlines(x, record.null_low, record.null_high, color="0.58", linewidth=2)
                    ax.scatter(
                        x,
                        record.null_mean,
                        s=34,
                        facecolors="white",
                        edgecolors="0.32",
                        zorder=3,
                    )
                    ax.scatter(
                        x,
                        record.real_max_persistence,
                        s=46,
                        color=REGION_COLORS[region],
                        edgecolors="black" if record.p_upper <= 0.05 else "white",
                        linewidth=1.0 if record.p_upper <= 0.05 else 0.5,
                        zorder=4,
                    )
                    ax.plot(
                        [x, x],
                        [record.null_mean, record.real_max_persistence],
                        color=REGION_COLORS[region],
                        alpha=0.3,
                        linewidth=0.8,
                    )
            ax.set_xticks(range(len(STIMULI)), [STIMULUS_LABELS[s] for s in STIMULI])
            ax.tick_params(axis="x", labelsize=8)
            ax.set_title(region)
            if col == 0:
                ax.set_ylabel(f"{method}\nMaximum persistence")
            ax.grid(axis="y", color="0.9", linewidth=0.7)

    legend = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor="0.25", label="Real"),
        Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor="white",
            markeredgecolor="0.35",
            label="Null mean (line: 95% interval)",
        ),
    ]
    fig.legend(handles=legend, loc="upper right", frameon=False)
    fig.suptitle(
        f"Real versus within-trial shift null: H{homology}, CEBRA versus PCA ({dim}D)",
        fontsize=15,
        fontweight="bold",
    )
    return fig


all_metrics = []
all_summaries = []
recording_figure_root = FIGURE_ROOT / "recording_barcodes"
recording_figure_root.mkdir(parents=True, exist_ok=True)

for dim in active_dims:
    metrics = load_barcode_metrics(dim)
    summary = recording_null_summary(metrics)
    metrics.to_csv(INTERMEDIATE_ROOT / f"barcode_metrics_{dim}d.csv", index=False)
    summary.to_csv(
        INTERMEDIATE_ROOT / f"barcode_recording_summary_{dim}d.csv", index=False
    )
    all_metrics.append(metrics)
    all_summaries.append(summary)

    condition_cols = ["region", "stimulus", "recording"]
    for _, condition in metrics.groupby(condition_cols, sort=True):
        fig = plot_recording_barcodes(condition, dim)
        identity = condition.iloc[0]
        stem = "__".join(
            safe_filename(value)
            for value in (identity.region, identity.stimulus, identity.recording, f"d{dim}")
        )
        fig.savefig(recording_figure_root / f"{stem}.png", dpi=250, bbox_inches="tight")
        if SAVE_RECORDING_PDF:
            fig.savefig(recording_figure_root / f"{stem}.pdf", bbox_inches="tight")
        plt.close(fig)

    for homology in range(MAX_HOMOLOGY + 1):
        fig = plot_regional_summary(summary, dim, homology)
        stem = f"barcode_null_summary_H{homology}_{dim}d"
        fig.savefig(FIGURE_ROOT / f"{stem}.png", dpi=300, bbox_inches="tight")
        fig.savefig(FIGURE_ROOT / f"{stem}.pdf", bbox_inches="tight")
        plt.show()

metrics_df = pd.concat(all_metrics, ignore_index=True)
barcode_summary_df = pd.concat(all_summaries, ignore_index=True)
metrics_df.to_csv(FIGURE_ROOT / "barcode_metrics_all_null_draws.csv", index=False)
barcode_summary_df.to_csv(
    FIGURE_ROOT / "barcode_max_persistence_recording_summary.csv", index=False
)

display(barcode_summary_df.head())
print(f"Barcode figures and tables saved to {FIGURE_ROOT}")

## Interpretation notes

- A recording is the independent observational unit. The 100 shuffles estimate its null distribution; they are not 100 biological replicates.
- `p_upper` is the finite-sample upper-tail empirical p-value `(1 + # null >= real) / 101` for maximum persistence.
- A highlighted real interval exceeds the 95th percentile of the largest interval in each null diagram, controlling the within-diagram search over bars; every barcode panel and the recording summary table report how many real features exceed that threshold.
- The representative null barcode is chosen near the median null maximum persistence for visualization; all 100 shuffles contribute to thresholds, intervals, and p-values.
- CEBRA and PCA use the same standardized neural matrix, sampled time indices, point count, scale normalization, kNN graph, and filtration construction.
- Reducers are fitted on real data and held fixed for null transforms. This tests whether coordinated temporal structure is needed to occupy the observed geometry under each learned representation.
- Each 1,000-sample trial is circularly shifted separately, preserving each neuron's within-trial values and autocorrelation while disrupting coordinated population timing without crossing trial boundaries.
- The 1,000-sample trial length is inferred from the dataset structure and validated by exact divisibility; confirm this assumption against the original preprocessing metadata before publication.
- Dimensions exceeding a recording's neuron count are excluded from both methods to keep comparisons paired.
- Reducer checkpoints, matched indices, PH diagrams, completion markers, cache inventory, configuration, and per-dimension metric tables are saved under the cache and barcode output directories.